In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

In [2]:
file_path = "../data/raw/Component_One_Core_Feasibility.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)

Dataset shape: (1200, 25)


In [3]:
y = df["feasibility_label"]

X = df.drop(columns=[
    "feasibility_label",
    "case_id",
    "business_description"
])

print("X:", X.shape)
print("y:", y.shape)

X: (1200, 22)
y: (1200,)


In [4]:
categorical_features = [
    "business_stage",
    "business_category",
    "district",
    "province",
    "location_type",
    "proposed_action",
    "competition_level"
]

numerical_features = [
    "available_capital_lkr",
    "loan_amount_lkr",
    "monthly_budget_lkr",
    "initial_inventory_cost_lkr",
    "expected_price_lkr",
    "expected_customers_per_day",
    "customer_demand_score",
    "expected_operating_days_per_month",
    "entrepreneur_experience_years",
    "location_suitability_score",
    "available_staff_count",
    "required_staff_count",
    "available_equipment_score",
    "required_equipment_score",
    "supplier_availability_score"
]

In [5]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (840, 22)
Validation: (180, 22)
Testing: (180, 22)


In [6]:
numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [7]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        ))
    ]
)

In [8]:
logistic_model.fit(X_train, y_train)

random_forest_model.fit(X_train, y_train)

print("Both models trained successfully.")

Both models trained successfully.


In [9]:
y_test_pred_lr = logistic_model.predict(X_test)
y_test_pred_rf = random_forest_model.predict(X_test)

In [10]:
def evaluate_model(model_name, y_true, y_pred):
    print("=" * 50)
    print(model_name)
    print("=" * 50)

    print("Accuracy:",
          round(accuracy_score(y_true, y_pred), 4))

    print("Weighted Precision:",
          round(
              precision_score(
                  y_true, y_pred,
                  average="weighted",
                  zero_division=0
              ), 4
          ))

    print("Weighted Recall:",
          round(
              recall_score(
                  y_true, y_pred,
                  average="weighted",
                  zero_division=0
              ), 4
          ))

    print("Weighted F1:",
          round(
              f1_score(
                  y_true, y_pred,
                  average="weighted",
                  zero_division=0
              ), 4
          ))

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            zero_division=0
        )
    )

In [11]:
evaluate_model(
    "Logistic Regression",
    y_test,
    y_test_pred_lr
)

Logistic Regression
Accuracy: 0.75
Weighted Precision: 0.7486
Weighted Recall: 0.75
Weighted F1: 0.7477

Classification Report:
                        precision    recall  f1-score   support

Conditionally Feasible       0.77      0.83      0.79       103
              Feasible       0.70      0.64      0.67        47
            Infeasible       0.77      0.67      0.71        30

              accuracy                           0.75       180
             macro avg       0.74      0.71      0.73       180
          weighted avg       0.75      0.75      0.75       180



In [12]:
evaluate_model(
    "Logistic Regression",
    y_test,
    y_test_pred_lr
)

Logistic Regression
Accuracy: 0.75
Weighted Precision: 0.7486
Weighted Recall: 0.75
Weighted F1: 0.7477

Classification Report:
                        precision    recall  f1-score   support

Conditionally Feasible       0.77      0.83      0.79       103
              Feasible       0.70      0.64      0.67        47
            Infeasible       0.77      0.67      0.71        30

              accuracy                           0.75       180
             macro avg       0.74      0.71      0.73       180
          weighted avg       0.75      0.75      0.75       180



In [13]:
evaluate_model(
    "Random Forest",
    y_test,
    y_test_pred_rf
)

Random Forest
Accuracy: 0.8833
Weighted Precision: 0.8846
Weighted Recall: 0.8833
Weighted F1: 0.8835

Classification Report:
                        precision    recall  f1-score   support

Conditionally Feasible       0.90      0.89      0.90       103
              Feasible       0.89      0.85      0.87        47
            Infeasible       0.82      0.90      0.86        30

              accuracy                           0.88       180
             macro avg       0.87      0.88      0.87       180
          weighted avg       0.88      0.88      0.88       180



In [14]:
print("Logistic Regression Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_lr))

print("\nRandom Forest Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_rf))

Logistic Regression Confusion Matrix:
[[85 12  6]
 [17 30  0]
 [ 9  1 20]]

Random Forest Confusion Matrix:
[[92  5  6]
 [ 7 40  0]
 [ 3  0 27]]


In [15]:
print("Logistic Regression Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_lr))

print("\nRandom Forest Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_rf))

Logistic Regression Confusion Matrix:
[[85 12  6]
 [17 30  0]
 [ 9  1 20]]

Random Forest Confusion Matrix:
[[92  5  6]
 [ 7 40  0]
 [ 3  0 27]]


In [16]:
print("===== LOGISTIC REGRESSION =====")
print(classification_report(y_test, y_test_pred_lr))

print("\n===== RANDOM FOREST =====")
print(classification_report(y_test, y_test_pred_rf))

===== LOGISTIC REGRESSION =====
                        precision    recall  f1-score   support

Conditionally Feasible       0.77      0.83      0.79       103
              Feasible       0.70      0.64      0.67        47
            Infeasible       0.77      0.67      0.71        30

              accuracy                           0.75       180
             macro avg       0.74      0.71      0.73       180
          weighted avg       0.75      0.75      0.75       180


===== RANDOM FOREST =====
                        precision    recall  f1-score   support

Conditionally Feasible       0.90      0.89      0.90       103
              Feasible       0.89      0.85      0.87        47
            Infeasible       0.82      0.90      0.86        30

              accuracy                           0.88       180
             macro avg       0.87      0.88      0.87       180
          weighted avg       0.88      0.88      0.88       180



In [17]:
from sklearn.metrics import roc_auc_score, average_precision_score

y_test_proba_lr = logistic_model.predict_proba(X_test)
y_test_proba_rf = random_forest_model.predict_proba(X_test)

print("Logistic Regression ROC-AUC:",
      roc_auc_score(y_test, y_test_proba_lr, multi_class="ovr", average="weighted"))

print("Random Forest ROC-AUC:",
      roc_auc_score(y_test, y_test_proba_rf, multi_class="ovr", average="weighted"))

print("\nLogistic Regression PR-AUC:",
      average_precision_score(
          y_test,
          y_test_proba_lr,
          average="weighted"
      ))

print("Random Forest PR-AUC:",
      average_precision_score(
          y_test,
          y_test_proba_rf,
          average="weighted"
      ))

Logistic Regression ROC-AUC: 0.8853237892711577
Random Forest ROC-AUC: 0.9599101035416826

Logistic Regression PR-AUC: 0.8579863477756013


Random Forest PR-AUC: 0.9494047658537137


In [18]:
print("===== LOGISTIC REGRESSION =====")
print(classification_report(y_test, y_test_pred_lr))

print("\n===== RANDOM FOREST =====")
print(classification_report(y_test, y_test_pred_rf))

===== LOGISTIC REGRESSION =====
                        precision    recall  f1-score   support

Conditionally Feasible       0.77      0.83      0.79       103
              Feasible       0.70      0.64      0.67        47
            Infeasible       0.77      0.67      0.71        30

              accuracy                           0.75       180
             macro avg       0.74      0.71      0.73       180
          weighted avg       0.75      0.75      0.75       180


===== RANDOM FOREST =====
                        precision    recall  f1-score   support

Conditionally Feasible       0.90      0.89      0.90       103
              Feasible       0.89      0.85      0.87        47
            Infeasible       0.82      0.90      0.86        30

              accuracy                           0.88       180
             macro avg       0.87      0.88      0.87       180
          weighted avg       0.88      0.88      0.88       180

